In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import chardet 


In [ ]:


with open("popular_anime.csv", "rb") as f:
    result = chardet.detect(f.read())


print(result)

df=pd.read_csv("popular_anime.csv",encoding="utf-8")
print(df.columns)

In [ ]:
emptydatapercentage=df.isnull().mean()*100
print(emptydatapercentage.sort_values(ascending=False))


In [ ]:
cols_to_drop=emptydatapercentage[emptydatapercentage>60].index
print(cols_to_drop)



In [ ]:
print(type(cols_to_drop))
print(cols_to_drop)


In [ ]:
print(df.columns.tolist())


In [ ]:
print(df.dtypes)

In [ ]:
df['aired_from'] = pd.to_datetime(df['aired_from'], errors='coerce')


# duration_per_ep'deki s?reyi dakikaya ?evir
df['duration_per_ep_minutes'] = df['duration_per_ep'].str.extract('(\\d+)').astype(float)

# Kategorik s?tunlar? d?n??t?r
df['status'] = df['status'].astype('category')
df['type'] = df['type'].astype('category')
df['rating'] = df['rating'].astype('category')
df['genres'] = df['genres'].astype('category')




In [ ]:
print(df.dtypes)

In [ ]:
print(emptydatapercentage.sort_values(ascending=False))

In [ ]:
df.drop(columns=['studios','producers','aired_to','image','trailer'],inplace=True)

In [ ]:
for col in ['genres', 'type', 'status', 'rating']:
    if col in df.columns and df[col].dtype.name == 'category':
        if 'Unknown' not in df[col].cat.categories:
            df[col] = df[col].cat.add_categories(['Unknown'])
    df[col] = df[col].fillna('Unknown')
for col in ['episodes', 'duration_per_ep_minutes','score','scored_by','rank','rating']:
    if col in df.columns and df[col].dtype.name == 'category':
        if '-1,0,23' not in df[col].cat.categories:
            df[col] = df[col].cat.add_categories([-1,0,23])
    

df.fillna({
    'name': 'Unknown',
    'genres': 'Unknown',
    'type': df['type'].mode()[0],
    'episodes': 23,
    'status': df['status'].mode()[0],
    'aired_from': pd.Timestamp('2000-01-01'),
    'aired_to': pd.Timestamp('2000-01-01'),
    'duration_per_ep_minutes': 24,
    'score': 0,
    'scored_by': df['scored_by'].median(),
    'rank': -1,
    'rating': 0,
    'image': 'No image',
    'trailer': 'No trailer',
    'synopsis': 'No synopsis available'
}, inplace=True)


In [ ]:
print(df.info())

In [ ]:
print(df['type'].value_counts())


In [ ]:
for col in df.select_dtypes(include=['object', 'category']):
    print(f"--- {col} ---")
    print(df[col].value_counts())
    print("\n")


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

In [ ]:
def clean_anime_name(name):
    name = re.sub(r"Season\s*\d+", "", name, flags=re.IGNORECASE)
    name = re.sub(r"Final\s*Season.*", "", name, flags=re.IGNORECASE)
    name = re.sub(r"Part\s*\d+", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\s*:\s*.*$", "", name, flags=re.IGNORECASE)  # isimden sonras?n? sil
    name = name.strip()
    return name


In [ ]:
# Dataset y?kle
df = pd.read_csv('popular_anime.csv')

# synopsis s?tununu doldur
df['synopsis'] = df['synopsis'].fillna('')


In [ ]:
df['name_clean'] = df['name'].apply(clean_anime_name)
df['name_lower'] = df['name'].str.lower()


In [ ]:
def map_type_group(x):
    if x in ['TV', 'OVA', 'ONA', 'Special']:
        return 'TV_GROUP'
    elif x == 'Movie':
        return 'MOVIE_GROUP'
    else:
        return 'OTHER'

df['type_group'] = df['type'].apply(map_type_group)


In [ ]:
# NaN isimleri sil
df_clean = df.dropna(subset=['name_clean'])

# en y?ksek skoru al
df_dedup = (
    df_clean
    .sort_values('score', ascending=False)
    .drop_duplicates(subset=['name_clean', 'type_group'], keep='first')
)

# TV ve Movie DataFrame'leri olu?tur
tv_final_df = df_dedup[df_dedup['type_group'] == 'TV_GROUP']
movie_final_df = df_dedup[df_dedup['type_group'] == 'MOVIE_GROUP']


In [ ]:
# TV synopsis listesi
tv_synopsis_list = tv_final_df['synopsis'].tolist()
# 1. Veri temizli?i (zaten yapm??s?n ama synopsis uzunluk filtresi ekleniyor)

df_filtered = df[df['synopsis'].str.len() >= 30].copy()
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df_filtered['synopsis'])

# TF-IDF vekt?rizer
vectorizer = TfidfVectorizer(stop_words='english')

# Movie synopsis listesi
movie_synopsis_list = movie_final_df['synopsis'].tolist()

vectorizer_movie = TfidfVectorizer(stop_words='english')
tfidf_matrix_movie = vectorizer_movie.fit_transform(movie_synopsis_list)


In [ ]:
!pip install rapidfuzz


In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from rapidfuzz import process, fuzz

In [ ]:

def clean_anime_name(name):
    if pd.isnull(name): return ""
    name = re.sub(r"Season\s*\d+", "", name, flags=re.IGNORECASE)
    name = re.sub(r"Final\s*Season.*", "", name, flags=re.IGNORECASE)
    name = re.sub(r"Part\s*\d+", "", name, flags=re.IGNORECASE)
    name = re.sub(r"[^a-zA-Z0-9\s]", "", name)
    return name.strip().lower()

def normalize(text): return text.strip().lower()

In [ ]:
# TV ve Movie veri setlerine g?venli bir ?ekilde 'source' ekle
tv_final_df = tv_final_df.copy()
tv_final_df["source"] = "tv"

movie_final_df = movie_final_df.copy()
movie_final_df["source"] = "movie"
remove_types = ["Special", "OVA", "ONA", "Music", "Recap"]

def clean_df(df):
    return df[
        df["synopsis"].notnull() &
        df["name"].notnull() &
        (df["score"].fillna(0) >= 5.0) &
        (~df["type"].isin(remove_types)) &
        (df["episodes"].fillna(2) > 1)
    ]

tv_final_df = clean_df(tv_final_df)
movie_final_df = clean_df(movie_final_df)

# Birle?tir ve yeni df_all'u garanti bir kopya olarak al
df_all = pd.concat([tv_final_df, movie_final_df], ignore_index=True).copy()

# G?venli ?ekilde temizleme i?lemleri
df_all["base_name"] = df_all["name"].apply(clean_anime_name).str.lower()

df_all["combined_text"] = (
    df_all["synopsis"].fillna("") + " " +
    df_all["genres"].fillna("") + " " +
    df_all["type"].fillna("") + " " +
    df_all["rating"].fillna("")
).str.lower()


In [ ]:
from rapidfuzz import process, fuzz

def fuzzy_merge(df_base, df_additional, on='base_name', threshold=90):
    matches = []
    for name in df_base[on]:
        match = process.extractOne(name, df_additional['English name cleaned'], scorer=fuzz.token_sort_ratio)
        if match and match[1] >= threshold:
            row = df_additional[df_additional['English name cleaned'] == match[0]]
            if not row.empty:
                row_data = {
                    "Score": row["Score"].values[0] if "Score" in row else np.nan,
                    "Rank": row["Rank"].values[0] if "Rank" in row else np.nan,
                    "Popularity": row["Popularity"].values[0] if "Popularity" in row else np.nan
                }
                matches.append(pd.Series(row_data))
            else:
                matches.append(pd.Series({"Score": np.nan, "Rank": np.nan, "Popularity": np.nan}))
        else:
            matches.append(pd.Series({"Score": np.nan, "Rank": np.nan, "Popularity": np.nan}))
    scores_df = pd.DataFrame(matches).reset_index(drop=True)
    return pd.concat([df_base.reset_index(drop=True), scores_df], axis=1)


In [ ]:
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix_all = vectorizer.fit_transform(df_all["combined_text"])
# anime-dataset-2023.csv dosyas?n? oku
df2 = pd.read_csv("anime-dataset-2023.csv")
df2["English name cleaned"] = df2["English name"].fillna(df2["Name"]).apply(clean_anime_name)
# Gereksiz kolonlar? sil
drop_cols = [
    "Producers", "Licensors", "Premiered",
    "Broadcast", "Opening Theme", "Ending Theme", "Background"
]
df_all.drop(columns=[col for col in drop_cols if col in df_all.columns], inplace=True)
# Merge et
df_all = fuzzy_merge(df_all, df2)
df_all["Score"] = pd.to_numeric(df_all["Score"], errors="coerce")

# Sat?r bazl? temizlik
df_all = df_all[df_all["synopsis"].notnull() & df_all["name"].notnull()]
df_all = df_all[df_all["Score"].fillna(0) > 4]






In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Gerekli kolonlar
required_cols = ["Score", "Rank", "Popularity"]

# Eksik olan kolonlar? ekle ve tipleri d?n??t?r
for col in required_cols:
    if col not in df_all.columns:
        df_all[col] = np.nan

# Tip kontrol?: Say?sal de?ilse d?n??t?r
for col in required_cols:
    df_all[col] = pd.to_numeric(df_all[col], errors='coerce')

# Ortalamayla doldur
df_all[required_cols] = df_all[required_cols].fillna(df_all[required_cols].mean())

# Normalize et
scaler = MinMaxScaler()
df_all[["norm_score", "norm_rank", "norm_popularity"]] = scaler.fit_transform(df_all[required_cols])

# Quality Score hesapla (score y?ksek, rank/popularity d???k iyidir)
df_all["quality_score"] = (
    df_all["norm_score"] * 0.5 +
    (1 - df_all["norm_rank"]) * 0.3 +
    (1 - df_all["norm_popularity"]) * 0.2
)


In [ ]:
from rapidfuzz import process
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# 1?? Kullan?lacak veriyi filtrele (?rne?in synopsis k?sa olanlar? at)
df_filtered = df_all[df_all["synopsis"].str.len() >= 30].copy()
df_filtered = df_filtered.reset_index(drop=True)  # index kar???kl???n? ?nlemek i?in s?f?rla

# 2?? TF-IDF matrisini olu?tur
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix_filtered = vectorizer.fit_transform(df_filtered["combined_text"])

# 3?? Kullan?c?dan anime isimleri al
print("?? L?tfen en az 1, en fazla 5 anime ismi giriniz.")
user_inputs = []
while len(user_inputs) < 5:
    anime_name = input(f"{len(user_inputs)+1}. anime ismi (bitirmek i?in bo? b?rak ve Enter'a bas): ").strip()
    if anime_name == '':
        break
    user_inputs.append(anime_name)

if not user_inputs:
    print("? En az bir anime ismi girmelisiniz.")
    exit()

# 4?? Anime isimlerini fuzzy match ile e?le?tir
matched_indices = []
for anime in user_inputs:
    cleaned_input = normalize(anime)
    best_match = process.extractOne(cleaned_input, df_filtered["base_name"].tolist(), score_cutoff=80)
    if best_match:
        idx = df_filtered[df_filtered["base_name"] == best_match[0]].index
        if not idx.empty:
            matched_indices.append(idx[0])
        print(f"? '{anime}' e?le?ti ? '{best_match[0]}'")
    else:
        print(f"?? '{anime}' i?in e?le?me bulunamad?.")

if not matched_indices:
    print("? Hi?bir ge?erli e?le?me bulunamad?.")
    exit()

# 5?? TF-IDF ortalama vekt?r?n? hesapla
input_vecs = tfidf_matrix_filtered[matched_indices]
mean_vec = input_vecs.mean(axis=0)
mean_vec = np.array(mean_vec)

# 6?? Cosine similarity ile benzerlikleri bul
cosine_sim = cosine_similarity(mean_vec, tfidf_matrix_filtered).flatten()

# 7?? Final skoru hesapla (benzerlik ve kalite skoru birlikte)
scored = []
for i, sim in enumerate(cosine_sim):
    qual = df_filtered.iloc[i]['quality_score'] if not pd.isna(df_filtered.iloc[i]['quality_score']) else 0
    final_score = 0.8 * sim + 0.2 * qual
    scored.append((i, final_score))


# 8?? En iyi 10 sonucu al (kullan?c?n?n girdiklerini ??kar)
top10 = sorted(scored, key=lambda x: x[1], reverse=True)
top10 = [item for item in top10 if item[0] not in matched_indices][:10]

pd.set_option("display.max_colwidth", None)
pd.set_option("display.expand_frame_repr", False)
pd.set_option("display.max_columns", None)
# 9?? ?nerileri yazd?r
print("\n?? Final Skor ile En ?yi 10 Anime ?nerisi:")
recommendations = df_filtered.iloc[[i for i, _ in top10]][['name', 'score', 'genres', 'quality_score']]
recommendations.reset_index(drop=True, inplace=True)
print(recommendations)


In [ ]:
print(df_all[['Score', 'Rank', 'Popularity']].head())


In [ ]:
df[df['synopsis'].str.len() < 30].shape

In [ ]:
print("TF-IDF shape:", tfidf_matrix_all.shape)
print("df_all shape:", df_all.shape)


In [ ]:
df_filtered.to_csv("anime_cleaned.csv", index=False)


In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
from rapidfuzz import process
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load your preprocessed anime dataset here
@st.cache_data
def load_data():
    df = pd.read_csv("anime_cleaned.csv")  # senin temizlenmi? df_filtered.csv
    df = df[df["synopsis"].str.len() >= 30].copy()
    df = df.reset_index(drop=True)
    return df

df = load_data()

# TF-IDF vectorizer
@st.cache_resource
def vectorize_text(df):
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(df['combined_text'])
    return vectorizer, tfidf_matrix

vectorizer, tfidf_matrix = vectorize_text(df)

# Normalizasyon
def normalize(text):
    return ''.join(e.lower() for e in text if e.isalnum() or e.isspace()).strip()

# Ba?l?k
st.title("?? Anime ?neri Sistemi")
st.write("Be?endi?iniz 1-5 animeyi girin, size benzeyenleri ?nerelim.")

# Giri? alan?
anime_inputs = []
for i in range(5):
    name = st.text_input(f"{i+1}. Anime Ad?", "")
    if name.strip():
        anime_inputs.append(name.strip())

# Buton
if st.button("?? ?nerileri Getir"):
    if not anime_inputs:
        st.warning("L?tfen en az 1 anime ismi girin.")
    else:
        matched_indices = []
        for anime in anime_inputs:
            cleaned_input = normalize(anime)
            match = process.extractOne(cleaned_input, df["base_name"].tolist(), score_cutoff=80)
            if match:
                idx = df[df["base_name"] == match[0]].index
                if not idx.empty:
                    matched_indices.append(idx[0])
                    st.success(f"? '{anime}' e?le?ti ? '{match[0]}'")
            else:
                st.warning(f"?? '{anime}' i?in e?le?me bulunamad?.")

        if not matched_indices:
            st.error("Hi?bir e?le?me bulunamad?.")
        else:
            # TF-IDF vekt?r?
            input_vecs = tfidf_matrix[matched_indices]
            mean_vec = input_vecs.mean(axis=0)
            mean_vec = np.array(mean_vec)

            # Benzerlik
            cosine_sim = cosine_similarity(mean_vec, tfidf_matrix).flatten()

            # Skor hesapla
            scored = []
            for i, sim in enumerate(cosine_sim):
                if i not in matched_indices:
                    qual = df.iloc[i]['quality_score'] if not pd.isna(df.iloc[i]['quality_score']) else 0
                    score = 0.8 * sim + 0.2 * qual
                    scored.append((i, score))

            # En iyi 10 ?neri
            top10 = sorted(scored, key=lambda x: x[1], reverse=True)[:10]
            results = df.iloc[[i for i, _ in top10]][['name', 'score', 'genres', 'quality_score']].reset_index(drop=True)

            st.subheader("?? En ?yi 10 ?neri:")
            st.dataframe(results)
